In [10]:
import numpy as np

class KNearestNeighbors:
    def __init__(self, k):
        if not isinstance(k, (int, np.integer)) or k <= 0:
            raise ValueError("K must be a positive integer")
        self.k = k
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=np.float64)
        self.y_train = np.asarray(y).ravel()

    def _euclidean_dist(self, x1, x2):
        return np.sqrt(np.sum((x1 - x2) ** 2))

    def _predict_one(self, x):

        distances = []
        for i, x_train in enumerate(self.X_train):
            dist = self._euclidean_dist(x, x_train)
            distances.append((dist, self.y_train[i]))

        distances.sort(key = lambda item: item[0])
        k_nearest = distances[:self.k]

        labels = [label for _, label in k_nearest]
        unique_labels, count = np.unique(labels, return_counts=True)

        print("Distances : ", distances)
        print("K nearest : ", k_nearest)
        print("Labels : ", labels)
        print("Unique labels : ", unique_labels)
        
        return unique_labels[np.argmax(count)]

    def predict(self, X):
        X = np.asarray(X, dtype=np.float64)
        return [self._predict_one(x) for x in X]




In [11]:
from ucimlrepo import fetch_ucirepo

def load_data(id: int):
    dataset = fetch_ucirepo(id = id)

    name = dataset.metadata.name
    X = dataset.data.features
    y = dataset.data.targets

    return name, X, y

In [12]:
def evaluate(y_act, y_pred):
    y_act = np.asarray(y_act).ravel()
    y_pred = np.asarray(y_pred).ravel()

    classes = np.unique(np.concatenate((y_act, y_pred)))
    C = len(classes)

    cm = np.zeros((C, C), dtype=int)

    classes_to_idx = {label: i for i, label in enumerate(classes)}

    for actual, predicted in zip(y_act, y_pred):
        cm[classes_to_idx[actual]][classes_to_idx[predicted]] += 1

    TP = np.diag(cm)
    FP = cm.sum(axis = 0) - TP
    FN = cm.sum(axis = 1) - TP

    precision = np.divide(
        TP, TP + FP,
        out = np.zeros(C, dtype=float),
        where = (TP + FP) != 0
    )

    recall = np.divide(
        TP, TP + FN,
        out = np.zeros(C, dtype=float),
        where = (TP + FN) != 0
    )

    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros(C, dtype=float),
        where=(precision + recall) != 0
    )


    return {
        'precision': precision,
        'recall': recall,
        'f1-score': f1,
        'cm': cm
    }



In [13]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import numpy as np

name, X, y = load_data(id = 53) 

# print(f"Exploratory Data Analysis for {name} dataset")
# eda(X, y)
# print("=" * 50)

print()
print("=" * 50)
print("KNN From Scratch")
print("=" * 50)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2, 
    random_state=42,
    stratify=y

)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = KNearestNeighbors(k = 3)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

y_test = np.asarray(y_test)

for key, value in evaluate(y_test, y_pred).items():
    print(f"{key}: {value}")


KNN From Scratch
Distances :  [(np.float64(0.21084769612988974), 'Iris-setosa'), (np.float64(0.23237546858888647), 'Iris-setosa'), (np.float64(0.34745176580262027), 'Iris-setosa'), (np.float64(0.4986699862033455), 'Iris-setosa'), (np.float64(0.49866998620334557), 'Iris-setosa'), (np.float64(0.5131752131903797), 'Iris-setosa'), (np.float64(0.5549623165678331), 'Iris-setosa'), (np.float64(0.575691295306706), 'Iris-setosa'), (np.float64(0.5997727449848703), 'Iris-setosa'), (np.float64(0.6003948496531548), 'Iris-setosa'), (np.float64(0.661412241161536), 'Iris-setosa'), (np.float64(0.661412241161536), 'Iris-setosa'), (np.float64(0.661412241161536), 'Iris-setosa'), (np.float64(0.7364864726665721), 'Iris-setosa'), (np.float64(0.8483323330890471), 'Iris-setosa'), (np.float64(0.943331968720834), 'Iris-setosa'), (np.float64(1.0341837532608509), 'Iris-setosa'), (np.float64(1.0754982842864347), 'Iris-setosa'), (np.float64(1.167423577839579), 'Iris-setosa'), (np.float64(1.1932610076768357), 'Iris-